# Study 957 — Holdco Discount 🏛️

**Buy a conglomerate below the sum of its parts — does the gap ever close?**

Some listed holding companies are almost pure wrappers around **one listed stake**. Christian
Dior is mostly LVMH. Heineken Holding is Heineken NV and nothing else. Liberty Broadband was
Charter. That makes them the rare case where you can mark net asset value **from the tape**:
multiply the stake's price by the number of shares the holdco owns, and compare it with what
the holdco itself costs. The gap is the **holdco discount**, and it is almost always there.

The pitch writes itself: you are buying EUR 1 of LVMH for 79 cents. The question this study
asks is the only one that matters — *does the missing 21 cents ever come back?*

We build the discount for **7 holdcos** from price-only closes,
2004-01-02 → 2026-06-30, and run two tests: does the gap mean-revert, and does buying it
wide — hedged, costed, with borrow charged on the short — pay.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `d51337b2a3bb`); the live
cells run the offline synthetic control only. As-of 2026-06-30.*


## 1. What the discount looks like

Seven holdcos, seven long-running gaps. Note how different they are — Heineken Holding sits quietly around 12%, Bollore drifted out past 70%. That range is the first clue: if these were *mistakes*, they would look alike, and they would close.

In [1]:
panel = [
    ('Heineken Holding',  'Heineken NV', 11.7,  4.2,   95),
    ('Christian Dior',    'LVMH',        14.7,  9.1,  232),
    ('Liberty Broadband', 'Charter',      6.2, 19.4,  184),
    ('Naspers',           'Tencent',     37.9, 19.5,  329),
    ('Prosus',            'Tencent',     36.3,  7.7,   94),
    ('Bollore',           'Vivendi',     74.4, 13.7, 1514),
    ('SoftBank',          'Alibaba',     59.9,  8.7,  105),
]
print('holdco             its main stake   avg gap   swing   half-life')
for h, s, mean, sd, hl in panel:
    print('%-18s %-14s %6.1f%% %6.1f%% %8d d' % (h, s, mean, sd, hl))

holdco             its main stake   avg gap   swing   half-life
Heineken Holding   Heineken NV      11.7%    4.2%       95 d
Christian Dior     LVMH             14.7%    9.1%      232 d
Liberty Broadband  Charter           6.2%   19.4%      184 d
Naspers            Tencent          37.9%   19.5%      329 d
Prosus             Tencent          36.3%    7.7%       94 d
Bollore            Vivendi          74.4%   13.7%     1514 d
SoftBank           Alibaba          59.9%    8.7%      105 d


> 🔬 **For the quants** — "half-life" is the AR(1) half-life of the raw discount. Bollore's is **1,514 trading days**: six years. A gap that takes six years to half-close is not a trade, it is a career.

## 2. Test one — does a wide gap narrow?

The clean way to ask it: when a holdco's discount is unusually wide *for that holdco* — a standard deviation above its own two-year average — what does the gap look like six months later? If the folklore is right, it should be narrower.

In [2]:
print('pooled slope on 24,572 observations : -0.0002')
print('  (negative would mean the gap closes)')
print('HAC t-statistic                     : -0.09')
print('R-squared                           : 0.0000')
print('names with the RIGHT sign           : 5 of 7')

pooled slope on 24,572 observations : -0.0002
  (negative would mean the gap closes)
HAC t-statistic                     : -0.09
R-squared                           : 0.0000
names with the RIGHT sign           : 5 of 7


**Nothing.** The slope is statistically invisible (*t* = -0.09, R² = 0.0000). Knowing that a holdco is unusually cheap against its own history tells you essentially nothing about where the gap will be in six months. **5 of 7** names lean the right way and two lean the wrong way — Christian Dior and Bollore both had wide gaps get *wider* — which is exactly what a coin flip looks like.

## 3. Test two — but does trading it pay anyway?

A gap does not have to be predictable *on average* to be tradable *at the extremes*. So: buy the holdco and short its stake one-for-one — that isolates the gap and nothing else — but only when the gap is unusually wide, and let go once it is back to normal. Charge 10 bps a leg, and 1% a year to borrow the short.

In [3]:
print('gross of costs                    : Sharpe +0.566  (t = +3.04)   <- looks like something')
print('after costs and borrow            : Sharpe +0.312  (t = +1.69)')
print('just owning the gap, never timing : Sharpe -0.268  (t = -1.41)')

gross of costs                    : Sharpe +0.566  (t = +3.04)   <- looks like something
after costs and borrow            : Sharpe +0.312  (t = +1.69)
just owning the gap, never timing : Sharpe -0.268  (t = -1.41)


Two things jump out. First, **just owning the discount and waiting lost money** — Sharpe -0.268, CAGR -1.68% a year across two decades. The gaps did not close; on balance they widened. Second, the *timed* version does look alive before costs. Which is where the real work starts.

## 4. The check that decides it

Three of our seven pairs trade as **thin over-the-counter ADRs** in New York — Naspers, Prosus and SoftBank. Their closing prints can be hours stale, or simply not refreshed at all. And a stale price on one leg of a long/short pair *invents* exactly the pattern we are hunting: an apparent gap today that "closes" tomorrow when the quote finally catches up.

So split the panel. Four pairs where both legs print in the same liquid session; three where one leg is a stale OTC quote.

In [4]:
rows = [
    ('full panel (all 7)', 0.566, 3.04, 0.312, 1.69),
    ('primary listings only (4)', 0.368, 1.87, 0.148, 0.76),
    ('the 3 thin OTC ADR pairs', 0.458, 2.16, 0.389, 1.85),
]
print('%-28s %9s %7s %9s %7s' % ('sub-panel', 'gross', '(t)', 'net', '(t)'))
for tag, gs, gt, ns, nt in rows:
    print('%-28s %+9.3f %+7.2f %+9.3f %+7.2f' % (tag, gs, gt, ns, nt))

sub-panel                        gross     (t)       net     (t)
full panel (all 7)              +0.566   +3.04    +0.312   +1.69
primary listings only (4)       +0.368   +1.87    +0.148   +0.76
the 3 thin OTC ADR pairs        +0.458   +2.16    +0.389   +1.85


There it is. Take away the stale-quote names and the gross *t* collapses from **+3.04 to +1.87**, and after costs to **+0.76**. What looked like a discount edge was largely the least liquid corner of the panel bouncing off its own stale prints.

## 5. And it was fragile anyway

Even taking the full-panel number at face value, it does not survive contact with a trading desk.

In [5]:
rows = [(0, 0.510, 2.75), (5, 0.411, 2.22), (10, 0.312, 1.69), (25, 0.016, 0.08), (50, -0.467, -2.52)]
print('cost per leg   ->  Sharpe (t)')
for c, sh, t in rows:
    flag = '   <- our assumption' if c == 10 else ''
    print('  %2d bps          %+.3f (%+.2f)%s' % (c, sh, t, flag))

cost per leg   ->  Sharpe (t)
   0 bps          +0.510 (+2.75)
   5 bps          +0.411 (+2.22)
  10 bps          +0.312 (+1.69)   <- our assumption
  25 bps          +0.016 (+0.08)
  50 bps          -0.467 (-2.52)


Flat at 25 bps a leg, -0.47 Sharpe at 50. This is a two-legged trade in OTC ADRs and a Paris small-cap; 10 bps is the *optimistic* end of the range. And no individual holdco clears the bar on its own — the best are Prosus (+0.66, *t* = +1.61) and SoftBank (+0.40, *t* = +1.17), which are, again, the two thinnest ADRs.

Then there is the honest question any backtest has to answer: *how much of this depends on the two dials we chose?* The rule says buy when the gap is one standard deviation wide and sell when it is back to normal. Those two numbers are arbitrary. So try every sensible pair of them.

In [6]:
print('how wide before we buy, how normal before we sell:')
print('  %d combinations tried' % 17)
print('  before costs, the t-stat runs from +1.56 to +3.43')
print('  AFTER costs, it runs from +0.08 to +1.90')
print('  combinations that clear the bar (t >= 2) after costs: 0 of 17')

how wide before we buy, how normal before we sell:
  17 combinations tried
  before costs, the t-stat runs from +1.56 to +3.43
  AFTER costs, it runs from +0.08 to +1.90
  combinations that clear the bar (t >= 2) after costs: 0 of 17


That is the cleanest statement in the study. Our reported setting is not the best one in the grid — a slightly lazier exit does better (+0.653, *t* = +3.43) — so nothing was cherry-picked. But **there is no setting of the dials at which this trade is significant after costs**. Not one of 17.

## 6. Is the harness even capable of finding this? (live, offline)

Fair question. So we build a synthetic world where the discount genuinely *does* mean-revert, and a second where it is a coin-flip random walk, and run the identical code on both. If the detector fires on the first and stays quiet on the second, then the flat answer above is a fact about holdcos, not a broken test.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from holdco_nav import data, strategy as st
planted = st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=957)[0])
null    = st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=957)[0])
print('a world where the gap DOES close : slope t = %+.2f, gross Sharpe %+.2f'
      % (planted['pooled_t'], planted['sharpe_timed_gross']))
print('a world where it is a coin flip  : slope t = %+.2f, gross Sharpe %+.2f'
      % (null['pooled_t'], null['sharpe_timed_gross']))

a world where the gap DOES close : slope t = -8.44, gross Sharpe +1.07
a world where it is a coin flip  : slope t = -0.71, gross Sharpe -0.09


The machinery works. It finds reversion when reversion is planted, and finds nothing when there is nothing. The real tape simply has nothing to find.

## Verdict

- **Signal — Weak.** The discount does **not** mean-revert: a gap one standard deviation wider than its own two-year norm predicts nothing six months out (*t* = -0.09, right sign in 5/7 names — a coin flip). The trading version looks alive gross (*t* = +3.04) but loses that once you charge realistic friction (*t* = +1.69) and again once you remove the three thin OTC ADRs whose stale closes manufacture the pattern (+1.87 gross, +0.76 net) — and it is significant after costs at **none** of the 17 entry/exit settings we tried.
- **Tradability — Mirage.** Flat at 25 bps a leg, negative at 50, a bootstrap confidence interval that includes zero, and a short leg you would have to borrow in exactly the illiquid names producing the number. Meanwhile the patient version — buy the discount and wait — lost 1.68% a year for 22 years.
- **What the gap really is.** Dividend tax leaking up the chain, a family voting block that makes a takeover impossible, holding-company overhead, and the risk that management reinvests your money badly. Those are reasons a thing is *worth* less, not reasons it is *priced wrong* — and they have no reason to go away on your schedule. Bollore's discount took six years to half-close, and it went the wrong way.